In [47]:
import pandas as pd

In [71]:
per_game = pd.read_csv(
    r"Datasets\Jugador_temporada - Basketball Reference\clean\nba_per_game_clean.csv"
)

advanced = pd.read_csv(
    r"Datasets\Jugador_temporada - Basketball Reference\clean\nba_advanced_clean.csv"
)

totals = pd.read_csv(
    r"Datasets\Jugador_temporada - Basketball Reference\clean\nba_totals_clean.csv"
)

shooting = pd.read_csv(
    r"Datasets\Jugador_temporada - Basketball Reference\clean\nba_shooting_clean.csv"
)

master = pd.read_csv(
    r"Datasets\Jugador_temporada - Basketball Reference\nba_master_dataset_v2.csv"
)

In [72]:
def build_main_team(df):

    records = []

    for (_, _, _), group in df.groupby(
        ["Player", "Age", "Season"],
        sort=False
    ):

        # eliminar filas agregadas (2TM, 3TM...)
        real = group[
            ~group["Team"]
            .astype(str)
            .str.match(r"^\dTM$")
        ].copy()

        if real.empty:
            continue

        # asegurar que G es numérico
        real["G"] = pd.to_numeric(
            real["G"],
            errors="coerce"
        )

        # equipo con más partidos
        team = real.loc[
            real["G"].idxmax(),
            "Team"
        ]

        records.append({

            "Player": group.iloc[0]["Player"],
            "Age": group.iloc[0]["Age"],
            "Season": group.iloc[0]["Season"],
            "MainTeam": team

        })

    return pd.DataFrame(records)

In [73]:
main_team = build_main_team(per_game)

In [74]:
print(main_team.shape)

print(
    main_team[
        ["Player","Age","Season"]
    ].duplicated().sum()
)

(12228, 4)
0


In [75]:
comparacion = main_team.merge(
    master[["Player", "Age", "Season"]],
    on=["Player", "Age", "Season"],
    how="left",
    indicator=True
)

comparacion[
    comparacion["_merge"] == "left_only"
]

,Player,Age,Season,MainTeam,_merge
3379,Marcus Williams,22.0,2007-08,NJN,left_only


In [76]:
master_team = master.copy()

In [77]:
master_team = master_team.drop(
    columns="Team"
)

In [78]:
master = master[
    master["Player"] != "League Average"
].copy()

master_team = master_team[
    master_team["Player"] != "League Average"
].copy()

In [79]:
master_team = (
    master
    .drop(columns="Team")
    .merge(
        main_team,
        on=["Player", "Age", "Season"],
        how="left"
    )
    .rename(columns={"MainTeam": "Team"})
)

In [80]:
print(master.shape)
print(master_team.shape)

print(master_team["Team"].isna().sum())

print(
    master_team[
        ["Player", "Age", "Season"]
    ].duplicated().sum()
)

(12227, 118)
(12227, 118)
0
0


### Guardamos el dataset maestro

In [81]:
master_team.to_csv(
    "nba_master_team_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

In [82]:
master_team.to_parquet(
    "nba_master_team_v1.parquet",
    index=False
)